## Actividad 3_23: Fine tuning de un modelo de lenguaje
<div style="border-style:groove;border-width:thin;padding:10px">
Basándonos en el ejemplo de fine tuning visto en IAGenerativa5, vais a hacer un trabajo en grupo que consistirá en los siguientes pasos:
<ol>
<li>Escoge un propósito y un dataset. Las posibilidades son infinitas. Con datasets presentes en HuggingFace puedes hacer: noticias de un tema concreto, recetas de cocina, letras de canciones de un género, reseñas de películas, código en un lenguaje de programación, etc. Pero incluso podrías tratar de hacer un dataset propio. Haciendo scraping de cualquier web puedes sacar textos. Solo los tienes que poner después en el formato correcto.</li>
<li>Escoge un modelo de Huging Face. Para ello tendrás que tener en cuenta las restricciones de hardware que tenemos y que queremos un resultado lo más preciso posible. Deberás hacer diversas pruebas con distintos modelos de lenguaje.</li>
<li>Realiza un entrenamiento y documenta:
<ul>
<li>Evolución del Training Loss y Validation Loss</li>
<li>Hiperparámetros elegidos y por qué</li>
<li>Tiempo dedicado al entrenamiento</li></ul></li>
<li>Prueba el modelo con, al menos, 5 prompts y comenta si los resultados tienen sentido con el propósito del fine-tuning.</li>
</ol>
</div>

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

import torch
if torch.cuda.is_available():
    device = "cuda" #la gráfica NVIDIA
elif torch.backends.mps.is_available():
    device = "mps" #Chip de apple (M1,M2,M3)
else:
    device = "cpu" #Al procesador normal

print(device)


/home/ciabd14/anaconda3/envs/llm-env2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [24]:
#Tu código
model_id = "HuggingFaceTB/SmolLM-360M"
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.pad_token = (
    tokenizer.eos_token
) # Esta linea es porque SmolLM no tiene un padding específico.
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1318.01it/s]


In [ ]:
# del model
# torch.cuda.empty_cache()
# del tokenizer
# torch.cuda.empty_cache()

In [25]:
from datasets import load_dataset

raw_datasets = load_dataset("sh0416/ag_news")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})

In [26]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'label': 3,
 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)',
 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."}

In [27]:
def filter_by_label(example):
    return example["label"] == 3

filtered_datasets = raw_datasets.filter(filter_by_label)
filtered_datasets = filtered_datasets.remove_columns("label")

In [28]:
def concatenate_fields(example):
    return {"text": example["title"] + " " + example["description"]}

filtered_datasets = filtered_datasets.map(concatenate_fields)

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
    )

tokenized_datasets = filtered_datasets.map(
    tokenize, batched=True, remove_columns=["title", "description"]
)

Map: 100%|██████████| 1900/1900 [00:00<00:00, 66326.90 examples/s]


Ahora vamos a cargar el data collector que es el componente que toma ejemplos individuales del dataset y los agrupa en batches listos para entrenar, gestionando el padding dinámicamente. 

In [29]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [30]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="business-news-generator",
    push_to_hub=False,
    per_device_train_batch_size=8,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    num_train_epochs=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=200,
)

In [31]:
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"].select(range(5000)),
    eval_dataset=tokenized_datasets["test"],
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss
200,2.758223,2.798961
400,2.505683,2.721800
600,2.396596,2.657909
800,1.791952,2.739969
1000,1.701668,2.740016
1200,1.693722,2.741253
1250,1.693722,2.741452


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


TrainOutput(global_step=1250, training_loss=2.1247297302246095, metrics={'train_runtime': 227.9861, 'train_samples_per_second': 43.862, 'train_steps_per_second': 5.483, 'total_flos': 1731726969984000.0, 'train_loss': 2.1247297302246095, 'epoch': 2.0})

In [34]:
from transformers import pipeline

pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer, device=device)

prompts = ["Google", "The stock market", "Gasoline prices", "Olympics games", "The world cup"]

for p in prompts:
    out = pipe(p, do_sample=True, temperature=0.1, max_new_tokens=40)[0]["generated_text"]
    print(f"Prompt: {p}\n→ {out}\n")

[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Google
→ Google #39;s IPO price set at \$85 US per share Google #39;s initial public offering price is set at \$85 US per share, excluding the



[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: The stock market
→ The stock market is in the midst of a bull run. The Dow Jones Industrial Average is up 18 points, the Nasdaq is up 12 points, and the S amp;P 5



[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Gasoline prices
→ Gasoline prices fall, but still high  NEW YORK (Reuters) - U.S. gasoline prices fell in July for the  first time in eight months, but still remained high, as  crude



[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Olympics games
→ Olympics games to boost UK economy The Olympics will boost the UK economy by 1.5 billion pounds (1.8 billion euros) and create 140,000 jobs, a report says

Prompt: The world cup
→ The world cup final is a test of unity The World Cup is a test of unity. The World Cup is a test of unity. The World Cup is a test of unity. The World Cup is a test of



In [35]:
from transformers import pipeline

pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer, device=device)

prompts = ["Manuel Llano", "Pablo Palacio"]

for p in prompts:
    out = pipe(p, do_sample=True, temperature=0.1, max_new_tokens=40)[0]["generated_text"]
    print(f"Prompt: {p}\n→ {out}\n")

[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Manuel Llano
→ Manuel Llano, a former Venezuelan president, is in talks to return to power after 16 years in exile. Manuel Llano, a former Venezuelan president, is in talks to return to

Prompt: Pablo Palacio
→ Pablo Palacio, a former Venezuelan president, is in talks to return to office after 18 years in exile.  CARACAS, Venezuela (Reuters) - Former Venezuelan President Hugo

